In [1]:
from retrievers.vector_retriever import VectorRetriever
from retrievers.fulltext_retriever import FullTextRetriever
from retrievers.hybrid_retriever import HybridRetriever
from retrievers.text2cyper import Text2CypherRetriever
from neo4j_manager import Neo4jManager
from agents.agentic_rag import AgenticRAG
from agents.multi_agent_system import MultiAgentSystem


In [2]:
neo4j = Neo4jManager()


In [ ]:
query = "How did Momotaro convince the animals to join him on his journey?"
return_fields = {
    "description": "description",
    "name": "name"
}
retriever = VectorRetriever(neo4j, "event_embeddings", return_fields)
results = retriever.retrieve(query)

print(f"Vector search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
query = "millet dumpling"
return_fields = {
    "description": "description",
    "name": "name"
}
retriever = FullTextRetriever(neo4j, "event_fulltext", "Event", "description", return_fields)
results = retriever.retrieve(query)

print(f"Fulltext search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
query = "millet dumpling"
retriever = HybridRetriever(neo4j, "event_embeddings", "event_fulltext", "Event", "description", vector_weight=0.0, return_fields=return_fields)
results = retriever.retrieve(query, 10)

print(f"Hybrid search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


In [ ]:
# What happens in Momotaro, in order, and which characters are involved in each event?
cypher_query = """
MATCH (f:Folktale {url: $url})-[:HAS_EVENT]->(e:Event)
OPTIONAL MATCH (e)-[:HAS_AGENT]->(a:Agent)
RETURN e.order AS order,
       e.name AS event,
       collect(a.name) AS agents
ORDER BY order
"""

params = {
    "url": "https://fairytalez.com/momotaro/"
}

results = neo4j.execute_query(cypher_query, params)

for idx, row in enumerate(results):
    print(f"- {idx}. {row["event"]}: {row["agents"]}")


In [ ]:
# Which characters are most important in the story based on how many events they appear in?
cypher_query = """
MATCH (a:Agent)
OPTIONAL MATCH (a)<-[:HAS_AGENT]-(e:Event)
RETURN a.name AS agent,
       count(DISTINCT e) AS event_count
ORDER BY event_count DESC
LIMIT 10
"""

results = neo4j.execute_query(cypher_query)

for idx, row in enumerate(results):
    print(f"- {row["agent"]}: {row["event_count"]}")



In [ ]:
# What happens after the specific event where Momotaro gives the dumpling to the dog?
cypher_query = """
MATCH (e:Event {name: $name})-[:POST_EVENT]->(next:Event)
RETURN e.name AS current_event,
       next.name AS next_event
"""

params = {
    "name": "Momotaro gives dog a dumpling."
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(f"- {row['current_event']} -> {row['next_event']}")


In [ ]:
# How many folktales come from Japan?
cypher_query = """
MATCH (f:Folktale)-[:FROM_NATION]->(n:Nation {name: $name})
RETURN count(f) AS total_folktales
"""

params = {
    "name": "japanese"
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(row["total_folktales"])


In [ ]:
# Who are Momotaro's friends?
cypher_query = """
MATCH (m:Agent {name: $name})-[r:RELATIONSHIP]->(a:Agent)
WHERE r.type = "friend"
RETURN a.name AS friend
"""

params = {
    "name": "Momotaro"
}

results = neo4j.execute_query(cypher_query, params)

for row in results:
    print(row["friend"])


In [ ]:
text2cypher = Text2CypherRetriever(neo4j)

text2cypher.add_few_shot_example(
    "What happens in Momotaro, in order, and which characters are involved in each event?",
    """
    MATCH (f:Folktale {url: "https://fairytalez.com/momotaro/"})-[:HAS_EVENT]->(e:Event)
    OPTIONAL MATCH (e)-[:HAS_AGENT]->(a:Agent)
    RETURN e.order AS order,
           e.name AS event,
           collect(a.name) AS agents
    ORDER BY order
    """
)

text2cypher.add_few_shot_example(
    "Which characters are most important in the story based on how many events they appear in?",
    """
    MATCH (a:Agent)
    OPTIONAL MATCH (a)<-[:HAS_AGENT]-(e:Event)
    RETURN a.name AS agent,
           count(DISTINCT e) AS event_count
    ORDER BY event_count DESC
    LIMIT 10
    """
)

text2cypher.add_few_shot_example(
    "How many folktales come from Japan?",
    """
    MATCH (f:Folktale)-[:FROM_NATION]->(n:Nation {name: "Japan"})
    RETURN count(f) AS total_folktales
    """
)

text2cypher.add_few_shot_example(
    "Who are Momotaro's friends?",
    """
    MATCH (m:Agent {name: "Momotaro"})-[r:RELATIONSHIP]->(a:Agent)
    WHERE r.type = "friend"
    RETURN a.name AS friend
    """
)

text2cypher.add_few_shot_example(
    "What happens after the specific event where Momotaro gives the dumpling to the dog?",
    """
    MATCH (e:Event {name: "Momotaro gives dog a dumpling."})-[:POST_EVENT]->(next:Event)
    RETURN e.name AS current_event,
           next.name AS next_event
    """
)


In [ ]:
# Consulta de conteo sobre nodo de dominio real
cypher, results = text2cypher.retrieve("How many Event nodes are in the graph?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")



In [ ]:
# Traversal: qué instituciones aparecen en el grafo
cypher, results = text2cypher.retrieve("How many characters have the role of helper?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")
    

In [3]:
rag = MultiAgentSystem(neo4j)


Node labels and properties:
:`Agent` {{id: STRING NOT NULL, name: STRING NOT NULL, description: STRING NOT NULL, race: STRING NOT NULL, ageGroup: STRING NOT NULL, gender: STRING NOT NULL}}
:`Object` {{id: STRING NOT NULL, name: STRING NOT NULL, description: STRING NOT NULL, type: STRING NOT NULL}}
:`Place` {{id: STRING NOT NULL, name: STRING NOT NULL, description: STRING NOT NULL, type: STRING NOT NULL}}
:`Event` {{id: STRING NOT NULL, name: STRING NOT NULL, embedding: LIST<FLOAT NOT NULL> NOT NULL, description: STRING NOT NULL, order: INTEGER NOT NULL, thoughts: LIST<STRING NOT NULL> NOT NULL}}
:`Genre` {{name: STRING NOT NULL, description: STRING NOT NULL}}
:`Nation` {{name: STRING NOT NULL, description: STRING NOT NULL}}
:`Role` {{name: STRING NOT NULL, description: STRING NOT NULL}}
:`Trait` {{name: STRING NOT NULL, description: STRING NOT NULL}}
:`Function` {{name: STRING NOT NULL, description: STRING NOT NULL}}
:`Folktale` {{url: STRING NOT NULL, title: STRING NOT NULL}}

Relatio

In [4]:
# Pregunta factual simple — el router elegirá vector o hybrid search
result = rag.answer("How many Event nodes are in the graph?", "test3")

print("\nQuestion:", result['question'])
print("Answer:", result['answer'])



========== ITERATION 1 ==========

  ========== QUESTION ==========
  How many Event nodes are in the graph?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "rag_tool",
    "args": {
      "queries": [
        "How many Event nodes are in the graph?"
      ]
    },
    "id": "91ce68c1-0779-4a53-9e1f-410affb68c4e",
    "type": "tool_call"
  }
]

========== ITERATION 1 ==========

  ========== QUESTION ==========
  How many Event nodes are in the graph?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "vector_search",
    "args": {
      "query": "Number of Event nodes in the graph"
    },
    "id": "055acd6b-6d6e-4b09-9591-4ce10850fde1",
    "type": "tool_call"
  }
]



    ========== TOOL -> RESPONSE (vector_search) ==========
    [
  {
    "type": "text",
    "text": "[\"The people looked on; five times, six times did he cross the flames, sprang out of the fire,\", \"“I’m in luck this morning,” said the dame, and she pulled the peach to shore with a split